# Medical Arabic Fine-tuning with Qwen2.5-3B

This notebook fine-tunes Qwen2.5-3B-Instruct on medical Arabic conversations using LoRA adapters.

## Features:
- ✅ Optimized for 8GB VRAM (RTX 3060)
- ✅ 4-bit quantization with LoRA
- ✅ Comprehensive error handling
- ✅ Automatic checkpoint saving
- ✅ Training/validation split
- ✅ Model testing after training
- ✅ Memory optimization

## Requirements:
```bash
pip install torch transformers datasets trl unsloth accelerate peft bitsandbytes
```

## 1️⃣ Environment Setup & GPU Verification

In [1]:
import os
import sys
import warnings
from pathlib import Path

# Suppress progress bars and warnings for cleaner output
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TQDM_DISABLE'] = '1'
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn

# Clear GPU cache if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("="*60)
print("🔧 ENVIRONMENT INFORMATION")
print("="*60)
print(f"Python Version  : {sys.version.split()[0]}")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU Name        : {gpu_props.name}")
    print(f"VRAM Total      : {gpu_props.total_memory / 1024**3:.1f} GB")
    print(f"CUDA Version    : {torch.version.cuda}")
    print(f"BF16 Supported  : {torch.cuda.is_bf16_supported()}")
    print("\n✅ GPU is ready! You can proceed.")
else:
    print("\n❌ [CRITICAL] PyTorch cannot detect the GPU!")
    print("Please check:")
    print("  1. NVIDIA drivers are installed")
    print("  2. CUDA toolkit is installed")
    print("  3. PyTorch was installed with CUDA support")
    raise RuntimeError("GPU not available. Cannot proceed with training.")

print("="*60)

🔧 ENVIRONMENT INFORMATION
Python Version  : 3.10.20
PyTorch Version : 2.6.0+cu124
CUDA Available  : True
GPU Name        : NVIDIA GeForce RTX 3070 Ti Laptop GPU
VRAM Total      : 8.0 GB
CUDA Version    : 12.4
BF16 Supported  : True

✅ GPU is ready! You can proceed.


## 2️⃣ Configuration Parameters

In [2]:
# ============================================================
# PAPERMILL PARAMETERS  (injected at runtime)
# ============================================================
DATASET_PATH  = "../data_pipeline/processed_data/chatml_dataset.json"
OUTPUT_DIR    = "outputs"
BASE_MODEL    = "Qwen/Qwen2.5-3B-Instruct"
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-4
BATCH_SIZE    = 8
GRAD_ACCUM    = 8


In [3]:
# Parameters
DATASET_PATH = "E:/FineTuning/services/service-medical-llm/training/experiments/data_pipeline/processed_data/chatml_dataset.json"
OUTPUT_DIR = "E:/FineTuning/services/service-medical-llm/training/outputs/test_sft"
BASE_MODEL = "unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit"
NUM_EPOCHS = 5
LEARNING_RATE = 0.0002
BATCH_SIZE = 8
GRAD_ACCUM = 8


In [4]:
# =============================================================================
# CONFIGURATION PARAMETERS
# =============================================================================

class Config:
    """Centralized configuration for the training pipeline"""
    
    # Model Configuration  (overridden by papermill BASE_MODEL parameter)
    MODEL_NAME = BASE_MODEL
    MAX_SEQ_LENGTH = 1024  # Reduced for 8GB VRAM
    LOAD_IN_4BIT = True   # Essential for VRAM efficiency
    
    # LoRA Configuration
    LORA_R = 16           # Increased from 8 for better performance
    LORA_ALPHA = 16       # Should match LORA_R
    LORA_DROPOUT = 0.05   # Small dropout for regularization
    
    # Training Configuration  (overridden by papermill parameters)
    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = GRAD_ACCUM
    LEARNING_RATE = LEARNING_RATE
    MAX_STEPS = 1000       # Increase for full training (e.g., 500-1000)
    WARMUP_STEPS = 10
    WEIGHT_DECAY = 0.01
    
    # Data Configuration  (overridden by papermill DATASET_PATH parameter)
    DATA_PATH = DATASET_PATH
    TRAIN_SPLIT = 0.95    # 95% train, 5% validation
    
    # Output Configuration  (overridden by papermill OUTPUT_DIR parameter)
    OUTPUT_DIR = OUTPUT_DIR
    CHECKPOINT_DIR = "checkpoints"
    FINAL_MODEL_DIR = "qwen_medical_arabic_lora"
    SAVE_STEPS = 20       # Save checkpoint every 20 steps
    LOGGING_STEPS = 5
    
    # Optimization
    USE_GRADIENT_CHECKPOINTING = "unsloth"  # Saves ~30% VRAM
    OPTIM = "adamw_8bit"  # 8-bit optimizer for VRAM efficiency
    
config = Config()

print("📋 Training Configuration:")
print(f"   Model: {config.MODEL_NAME}")
print(f"   Max Sequence Length: {config.MAX_SEQ_LENGTH}")
print(f"   LoRA Rank: {config.LORA_R}")
print(f"   Effective Batch Size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")
print(f"   Max Training Steps: {config.MAX_STEPS}")
print(f"   Learning Rate: {config.LEARNING_RATE}")
print(f"   Dataset Path: {config.DATA_PATH}")
print(f"   Output Dir: {config.OUTPUT_DIR}")


📋 Training Configuration:
   Model: unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit
   Max Sequence Length: 1024
   LoRA Rank: 16
   Effective Batch Size: 8
   Max Training Steps: 1000
   Learning Rate: 0.0002
   Dataset Path: E:/FineTuning/services/service-medical-llm/training/experiments/data_pipeline/processed_data/chatml_dataset.json
   Output Dir: E:/FineTuning/services/service-medical-llm/training/outputs/test_sft


## 3️⃣ Load Base Model with LoRA Adapters

In [5]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("\n" + "="*60)
print("📥 LOADING BASE MODEL")
print("="*60)

try:
    # Load base model with 4-bit quantization
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.MODEL_NAME,
        max_seq_length=config.MAX_SEQ_LENGTH,
        dtype=None,  # Auto-detect (bf16 for RTX 30xx, fp16 for older)
        load_in_4bit=config.LOAD_IN_4BIT,
        device_map="auto",  # Automatic GPU distribution
        trust_remote_code=True,
    )
    
    print(f"✅ Successfully loaded: {config.MODEL_NAME}")
    print(f"📦 VRAM Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
except Exception as e:
    print(f"❌ Error loading model: {str(e)}")
    raise

print("\n" + "-"*60)
print("🔧 Attaching LoRA Adapters")
print("-"*60)

try:
    # Apply LoRA adapters to the model
    model = FastLanguageModel.get_peft_model(
        model,
        r=config.LORA_R,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=config.LORA_ALPHA,
        lora_dropout=config.LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing=config.USE_GRADIENT_CHECKPOINTING,
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    
    # Calculate trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_percentage = 100 * trainable_params / total_params
    
    print(f"✅ LoRA adapters attached successfully")
    print(f"📊 Trainable Parameters: {trainable_params:,} / {total_params:,}")
    print(f"📈 Trainable Percentage: {trainable_percentage:.2f}%")
    print(f"💾 Current VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
except Exception as e:
    print(f"❌ Error attaching LoRA adapters: {str(e)}")
    raise

print("="*60)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!



📥 LOADING BASE MODEL


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3070 Ti Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ Successfully loaded: unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit
📦 VRAM Allocated: 2.21 GB

------------------------------------------------------------
🔧 Attaching LoRA Adapters
------------------------------------------------------------


Unsloth 2026.4.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA adapters attached successfully
📊 Trainable Parameters: 29,933,568 / 1,830,055,936
📈 Trainable Percentage: 1.64%
💾 Current VRAM: 2.32 GB


## 4️⃣ Setup Chat Template & Load Dataset

In [6]:
from datasets import load_dataset, DatasetDict

print("\n" + "="*60)
print("💬 SETTING UP CHAT TEMPLATE")
print("="*60)

try:
    # Configure ChatML template
    tokenizer = get_chat_template(
        tokenizer,
        chat_template="chatml",
        mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
    )
    print("✅ ChatML template configured successfully")
    
except Exception as e:
    print(f"❌ Error setting up chat template: {str(e)}")
    raise

def formatting_prompts_func(examples):
    """Format conversations using the ChatML template"""
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

print("\n" + "-"*60)
print("📂 LOADING DATASET")
print("-"*60)

try:
    # Check if data file exists
    data_path = Path(config.DATA_PATH)
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found at: {data_path}")
    
    # Load dataset
    dataset = load_dataset(
        "json",
        data_files=str(data_path),
        split="train"
    )
    
    print(f"✅ Dataset loaded: {len(dataset)} examples")
    
    # Apply formatting
    dataset = dataset.map(
        formatting_prompts_func, 
        batched=True,
        desc="Formatting dataset"
    )
    
    # Split into train and validation
    split_dataset = dataset.train_test_split(
        test_size=1 - config.TRAIN_SPLIT,
        seed=3407
    )
    
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
    
    print(f"📊 Train Examples: {len(train_dataset)}")
    print(f"📊 Validation Examples: {len(eval_dataset)}")
    
    # Show sample
    print("\n--- Sample Formatted Data ---")
    sample_text = train_dataset[0]['text']
    print(sample_text[:400] + "..." if len(sample_text) > 400 else sample_text)
    
except FileNotFoundError as e:
    print(f"❌ {str(e)}")
    print("Please ensure the dataset is in the correct location.")
    raise
except Exception as e:
    print(f"❌ Error loading dataset: {str(e)}")
    raise

print("="*60)

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.



💬 SETTING UP CHAT TEMPLATE


✅ ChatML template configured successfully

------------------------------------------------------------
📂 LOADING DATASET
------------------------------------------------------------


Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded: 2000 examples


Formatting dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

📊 Train Examples: 1899
📊 Validation Examples: 101

--- Sample Formatted Data ---
<|im_start|>system
أنت طبيب نفسي عربي خبير. تجيب على استشارات المرضى بتعاطف، واحترافية، ودقة طبية عالية.<|im_end|>
<|im_start|>user
السلام عليكم.

أعاني منذ سنوات من أعراض دائمة لا أعرف سببها، تزيد أو تقل حدتها في بعض الأوقات، لكنها لا تتركني أبدًا.

وهذه الأعراض كالتالي: خمول وإرهاق دائمين، وشعور بالرغبة في النوم بشكل دائم، أحيانًا أستيقظ من النوم، فأجد سوادًا تحت العينين وكأنني لم أنم منذ أيام، ...


## 5️⃣ Initialize Trainer with Optimized Settings

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

print("\n" + "="*60)
print("🏋️ INITIALIZING TRAINER")
print("="*60)

# Create output directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

# Custom callback for better logging
class MemoryLoggingCallback(TrainerCallback):
    """Custom callback to log memory usage during training"""
    
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % config.LOGGING_STEPS == 0:
            allocated = torch.cuda.memory_allocated(0) / 1024**3
            reserved = torch.cuda.memory_reserved(0) / 1024**3
            print(f"Step {state.global_step}: VRAM Allocated={allocated:.2f}GB, Reserved={reserved:.2f}GB")

try:
    # Initialize trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,  # Added validation dataset
        dataset_text_field="text",
        max_seq_length=config.MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=False,  # Set to True if you have many short sequences
        args=TrainingArguments(
            # Batch and Accumulation
            per_device_train_batch_size=config.BATCH_SIZE,
            per_device_eval_batch_size=config.BATCH_SIZE,
            gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
            
            # Learning Rate and Schedule
            learning_rate=config.LEARNING_RATE,
            warmup_steps=config.WARMUP_STEPS,
            lr_scheduler_type="cosine",  # Changed from linear for better convergence
            
            # Training Steps
            max_steps=config.MAX_STEPS,
            
            # Precision
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            
            # Optimizer
            optim=config.OPTIM,
            weight_decay=config.WEIGHT_DECAY,
            
            # Logging and Saving
            logging_steps=config.LOGGING_STEPS,
            save_steps=config.SAVE_STEPS,
            save_total_limit=3,  # Keep only last 3 checkpoints
            eval_strategy="steps",  # Evaluate during training
            eval_steps=config.SAVE_STEPS,
            load_best_model_at_end=True,  # Load best checkpoint at end
            metric_for_best_model="eval_loss",
            
            # Output
            output_dir=config.CHECKPOINT_DIR,
            report_to="none",
            
            # Stability
            seed=3407,
            data_seed=3407,
            
            # Memory Optimization
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        ),
        callbacks=[MemoryLoggingCallback()],
    )
    
    print("✅ SFTTrainer initialized successfully!")
    print(f"📊 Training will run for {config.MAX_STEPS} steps")
    print(f"💾 Checkpoints will be saved every {config.SAVE_STEPS} steps")
    print(f"📈 Validation will run every {config.SAVE_STEPS} steps")
    
except Exception as e:
    print(f"❌ Error initializing trainer: {str(e)}")
    raise

print("="*60)


🏋️ INITIALIZING TRAINER


Unsloth: Tokenizing ["text"]:   0%|          | 0/1899 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/101 [00:00<?, ? examples/s]

✅ SFTTrainer initialized successfully!
📊 Training will run for 1000 steps
💾 Checkpoints will be saved every 20 steps
📈 Validation will run every 20 steps


## 6️⃣ Start Training with Memory Monitoring

In [8]:
import time

print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)

# Display pre-training memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)

print(f"\n[Pre-Training Stats]")
print(f"  GPU: {gpu_stats.name}")
print(f"  Total VRAM: {max_memory} GB")
print(f"  Reserved VRAM: {start_gpu_memory} GB")
print(f"  Available VRAM: {max_memory - start_gpu_memory} GB")
print("\n" + "-"*60)

try:
    # Start training with error handling
    start_time = time.time()
    trainer_stats = trainer.train()
    end_time = time.time()
    
    # Calculate training statistics
    used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
    used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
    training_time = end_time - start_time
    
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY!")
    print("="*60)
    print(f"⏱️  Total Runtime: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
    print(f"📊 Total Steps: {trainer_stats.global_step}")
    print(f"📉 Final Training Loss: {trainer_stats.training_loss:.4f}")
    print(f"💾 Peak VRAM Used: {used_memory_for_lora} GB")
    print(f"💾 Total VRAM Reserved: {used_memory} GB")
    
    if hasattr(trainer_stats, 'metrics'):
        print("\n📈 Training Metrics:")
        for key, value in trainer_stats.metrics.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
    
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted by user")
    print("Saving current checkpoint...")
    trainer.save_model(config.CHECKPOINT_DIR + "/interrupted")
    print(f"✅ Checkpoint saved to: {config.CHECKPOINT_DIR}/interrupted")
    
except RuntimeError as e:
    if "out of memory" in str(e):
        print("\n❌ CUDA Out of Memory Error!")
        print("\nSuggestions:")
        print("  1. Reduce MAX_SEQ_LENGTH (currently {config.MAX_SEQ_LENGTH})")
        print("  2. Reduce LORA_R (currently {config.LORA_R})")
        print("  3. Increase GRADIENT_ACCUMULATION_STEPS")
        print("  4. Close other GPU-using applications")
    raise
    
except Exception as e:
    print(f"\n❌ Error during training: {str(e)}")
    raise

print("="*60)


🚀 STARTING TRAINING

[Pre-Training Stats]
  GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
  Total VRAM: 8.0 GB
  Reserved VRAM: 3.383 GB
  Available VRAM: 4.617 GB

------------------------------------------------------------


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,899 | Num Epochs = 5 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,2.055600,1.994934


Step 5: VRAM Allocated=2.40GB, Reserved=3.60GB


Step 10: VRAM Allocated=2.40GB, Reserved=3.60GB


Step 15: VRAM Allocated=2.40GB, Reserved=3.60GB


Step 20: VRAM Allocated=2.40GB, Reserved=3.60GB


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step 25: VRAM Allocated=2.40GB, Reserved=3.60GB


## 7️⃣ Save Final Model and Adapters

In [ ]:
print("\n" + "="*60)
print("💾 SAVING FINAL MODEL")
print("="*60)

try:
    # Create output directory
    os.makedirs(config.FINAL_MODEL_DIR, exist_ok=True)
    
    # Save LoRA adapters (smaller, recommended for deployment)
    print(f"\nSaving LoRA adapters to: {config.FINAL_MODEL_DIR}")
    model.save_pretrained(config.FINAL_MODEL_DIR)
    tokenizer.save_pretrained(config.FINAL_MODEL_DIR)
    
    print("\n✅ Model saved successfully!")
    print(f"\n📦 Output Directory: {config.FINAL_MODEL_DIR}/")
    print("\nSaved Files:")
    
    total_size = 0
    for file_path in Path(config.FINAL_MODEL_DIR).rglob("*"):
        if file_path.is_file():
            file_size = file_path.stat().st_size / 1024**2  # MB
            total_size += file_size
            print(f"  - {file_path.name}: {file_size:.1f} MB")
    
    print(f"\n📊 Total Size: {total_size:.1f} MB")
    
    # Optional: Save merged model (much larger, but standalone)
    # Uncomment if you want a fully merged model
    # print("\n" + "-"*60)
    # print("Saving merged model (this will take a while)...")
    # merged_model_dir = config.FINAL_MODEL_DIR + "_merged"
    # model.save_pretrained_merged(merged_model_dir, tokenizer, save_method="merged_16bit")
    # print(f"✅ Merged model saved to: {merged_model_dir}")
    
except Exception as e:
    print(f"\n❌ Error saving model: {str(e)}")
    raise

print("\n" + "="*60)

## 8️⃣ Test the Fine-tuned Model

In [ ]:
print("\n" + "="*60)
print("🧪 TESTING FINE-TUNED MODEL")
print("="*60)

# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompts (Arabic medical questions)
test_prompts = [
    "أشعر بحزن شديد وفقدان للشغف في الفترة الأخيرة، ماذا أفعل؟",
    "كيف يمكنني التعامل مع نوبات الهلع والقلق المفاجئة؟",
    "ما هي علامات الاكتئاب السريري وكيف أعرف إذا كنت أحتاج لمساعدة طبيب؟",
]

print("\nGenerating responses to test prompts...\n")

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"Test {i}/{len(test_prompts)}")
    print(f"{'='*60}")
    print(f"\n🙋 User: {prompt}")
    
    # Format the prompt using chat template
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
    # Generate response
    try:
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        
        # Decode and print response
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the assistant's response
        if "<|im_start|>assistant" in response:
            assistant_response = response.split("<|im_start|>assistant")[-1]
            assistant_response = assistant_response.replace("<|im_end|>", "").strip()
            print(f"\n🤖 Assistant: {assistant_response}")
        else:
            print(f"\n🤖 Assistant: {response}")
            
    except Exception as e:
        print(f"\n❌ Error generating response: {str(e)}")

print("\n" + "="*60)
print("✅ Testing complete!")
print("="*60)